In [0]:
# Databricks notebook source

# COMMAND ----------
# Formula 1 Racing Data Platform
# Notebook: 01_ingest_bronze
#
# Purpose:
#   Read all raw CSV files and ingest them into the Bronze layer.

In [0]:
%run ../common/00_configure_catalog

In [0]:
# COMMAND ----------
# List Raw Files

files = dbutils.fs.ls(RAW_PATH)

print(f"Found {len(files)} files")

In [0]:
# COMMAND ----------
# Helper Functions

def read_csv(file_name: str):
    return (
        spark.read
             .option("header", True)
             .option("inferSchema", True)
             .csv(f"{RAW_PATH}/{file_name}")
    )

In [0]:
# COMMAND ----------
# Bronze Ingestion

for file in files:

    table_name = file.name.removesuffix(".csv")

    print(f"⚙️ Ingesting {table_name}")

    df = read_csv(file.name)

    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", True)
          .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")
    )
print("")
print("✅ Bronze ingestion completed.")

In [0]:
# COMMAND ----------
# Validation

spark.sql(f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}").display()